In [ ]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
!pip install -e .

# Building the prompts dataset

In [ ]:
import requests

prompts = None

api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
if api_response.ok:
    prompts = api_response.json()
else:
    print(api_response.text)

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [ ]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

117

### Get the dataset prompts

In [ ]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: 'emotone_ar' in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(filtered_prompts)

117

### Download the dataset

In [ ]:
import datasets

In [ ]:
emotone_ar_experimental = datasets.load_dataset('KFUPM-JRCAI/emotone_ar_experimental')
emotone_ar_experimental

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 8052
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 2013
    })
})

### Merge the prompts

In [ ]:
from jinja2 import Environment, StrictUndefined

In [ ]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    if "|||" not in template:
        raise ValueError("No ||| dividor")
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

see how the template is applied on different examples

### Perform generation on one example prompt, for experimentation

In [ ]:
example_prompt_template = dataset_prompts[4]
print(apply_template(example_prompt_template, emotone_ar_experimental['train'][2]))

Review this tweet المشكله ليست فيمن يخذلك ، يخونك ، يوجعك ، يسحقك ، المشكله هي انك تتمكن من تصديق شخص نال منك مره لتمنحه فرصه النيل منك اخري ! carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear.
|||
sadness


In [ ]:
rendered_test_prompts_dataset = list(map(lambda sample: apply_template(example_prompt_template, sample), emotone_ar_experimental['test']))
len(rendered_test_prompts_dataset)

2013

In [ ]:
from datasets import DatasetDict, load_dataset
def create_hf_dataset(examples, columns = ['tweet', 'emotion']):
  tweets = []
  emotions = []
  for example in examples:
    input= example.split('|||')[0].replace('\n', '')
    output = example.split('|||')[1].replace('\n', '')
    tweets.append(input)
    emotions.append(output)
  dataset = DatasetDict( { 'test' : datasets.Dataset.from_dict({
      columns[0]: tweets,
      columns[1]: emotions,
  })})
  return dataset

dataset = create_hf_dataset(rendered_test_prompts_dataset)
os.makedirs('emotone_dataset', exist_ok=True)
dataset['test'].to_parquet("emotone_dataset/data.parquet")

# later
ds = load_dataset("emotone_dataset")

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Generating train split: 0 examples [00:00, ? examples/s]

## Evaluate the LLM

In [ ]:
import os
yaml_text = '''task: emotone
dataset_path: emotone_dataset
output_type: multiple_choice
test_split: train
doc_to_text:  tweet
doc_to_target: emotion
doc_to_choice:
  - "none"
  - "anger"
  - "joy"
  - "sadness"
  - "love"
  - "sympathy"
  - "surprise"
  - "fear"
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''

def save_yaml(yaml_text):
  os.makedirs('lm_eval/tasks/emotone', exist_ok=True)
  with open('lm_eval/tasks/emotone/emotone.yaml', 'w') as f:
    f.write(yaml_text)

save_yaml(yaml_text)

In [ ]:
!cat lm_eval/tasks/emotone/emotone.yaml

task: emotone
dataset_path: emotone_dataset
output_type: multiple_choice
test_split: train
doc_to_text:  tweet
doc_to_target: emotion
doc_to_choice:
  - "none"
  - "anger"
  - "joy"
  - "sadness"
  - "love"
  - "sympathy"
  - "surprise"
  - "fear"
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=inception-mbzuai/jais-13b,trust_remote_code=True \
    --tasks emotone \
    --device cuda:0 \
    --batch_size 8


pytorch_model-00003-of-00006.bin:  35% 3.45G/9.96G [02:31<04:30, 24.1MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.46G/9.96G [02:31<04:30, 24.1MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.47G/9.96G [02:32<04:28, 24.2MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.48G/9.96G [02:32<04:27, 24.2MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.49G/9.96G [02:32<04:27, 24.2MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.50G/9.96G [02:33<04:26, 24.2MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.51G/9.96G [02:33<04:25, 24.3MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.52G/9.96G [02:34<04:24, 24.3MB/s]
pytorch_model-00003-of-00006.bin:  35% 3.53G/9.96G [02:34<04:24, 24.3MB/s]
pytorch_model-00003-of-00006.bin:  36% 3.54G/9.96G [02:35<04:23, 24.3MB/s]
pytorch_model-00003-of-00006.bin:  36% 3.55G/9.96G [02:35<04:23, 24.4MB/s]
pytorch_model-00003-of-00006.bin:  36% 3.57G/9.96G [02:36<04:22, 24.4MB/s]
pytorch_model-00003-of-00006.bin:  36% 3.58G/9.96G [02:36<04:22, 24.4MB/s]
pytorch_model-00003-of-0